In [1]:
# 1 — KAGGLE ENVIRONMENT + GLOBAL PATH REGISTRY (FINAL)

import os
import gc
import numpy as np
import pandas as pd

# --- FIND COMPETITION ROOT ---
BASE_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = None
for p in BASE_CANDIDATES:
    if os.path.exists(p):
        COMP_ROOT = p
        break

if COMP_ROOT is None:
    raise FileNotFoundError(
        "❌ Competition dataset not found.\n"
        "👉 Click 'Add Input' and attach: Stanford RNA 3D Folding 2"
    )

# --- INPUT FILES ---
TRAIN_LABELS_PATH = os.path.join(COMP_ROOT, "train_labels.csv")
VALIDATION_LABELS_PATH = os.path.join(COMP_ROOT, "validation_labels.csv")
TRAIN_SEQS_PATH = os.path.join(COMP_ROOT, "train_sequences.csv")
VALIDATION_SEQS_PATH = os.path.join(COMP_ROOT, "validation_sequences.csv")
TEST_SEQS_PATH = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_PATH = os.path.join(COMP_ROOT, "sample_submission.csv")

REQUIRED_FILES = [
    TRAIN_LABELS_PATH,
    VALIDATION_LABELS_PATH,
    TRAIN_SEQS_PATH,
    VALIDATION_SEQS_PATH,
    TEST_SEQS_PATH,
    SAMPLE_SUB_PATH,
]

missing_files = [p for p in REQUIRED_FILES if not os.path.exists(p)]
if missing_files:
    raise FileNotFoundError("❌ Missing required files:\n" + "\n".join(missing_files))

# --- WORKING OUTPUT PATHS ---
WORK_DIR = "/kaggle/working"
os.makedirs(WORK_DIR, exist_ok=True)

FEATURE_OUT = os.path.join(WORK_DIR, "FEATURE_TABLE__GEOMETRY_LABELS_V1.csv")
BENCH_OUT   = os.path.join(WORK_DIR, "HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv")

# --- PRINT STATE ---
print("✅ Competition root:", COMP_ROOT)
print("✅ Working dir:", WORK_DIR)
print("✅ Feature output:", FEATURE_OUT)
print("✅ Benchmark output:", BENCH_OUT)

print("\n📁 Files in competition root:")
for f in os.listdir(COMP_ROOT):
    print(" -", f)

✅ Competition root: /kaggle/input/competitions/stanford-rna-3d-folding-2
✅ Working dir: /kaggle/working
✅ Feature output: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv
✅ Benchmark output: /kaggle/working/HELIX_GEOMETRY_RECON_BENCHMARK_V1.csv

📁 Files in competition root:
 - MSA
 - sample_submission.csv
 - validation_sequences.csv
 - test_sequences.csv
 - validation_labels.csv
 - extra
 - train_labels.csv
 - train_sequences.csv
 - PDB_RNA


In [2]:
# 2 — LOAD COMPETITION LABELS (MEMORY-SAFE, OPTIMIZED)

import pandas as pd
import numpy as np
import gc

USE_COLS = ["ID", "resname", "resid", "x_1", "y_1", "z_1", "chain", "copy"]

DTYPES = {
    "ID": "string",
    "resname": "category",
    "resid": "int32",
    "x_1": "float32",
    "y_1": "float32",
    "z_1": "float32",
    "chain": "category",
    "copy": "category",
}

print("📥 Loading train labels...")
df_labels = pd.read_csv(
    TRAIN_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

print("📥 Appending validation labels...")
df_val = pd.read_csv(
    VALIDATION_LABELS_PATH,
    usecols=USE_COLS,
    dtype=DTYPES,
    low_memory=False
)

# Append WITHOUT creating large intermediate copies
df_labels = pd.concat([df_labels, df_val], ignore_index=True)
del df_val
gc.collect()

# --- DERIVE FIELDS ---
df_labels["target_id"] = df_labels["ID"].str.split("_").str[0]

df_labels["residue_index"] = df_labels["resid"].astype(np.int32)

df_labels["x"] = df_labels["x_1"]
df_labels["y"] = df_labels["y_1"]
df_labels["z"] = df_labels["z_1"]

# Drop original heavy columns EARLY
df_labels = df_labels.drop(columns=["ID", "resid", "x_1", "y_1", "z_1"])

gc.collect()

# --- SORT (critical for geometry) ---
df_labels = df_labels.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

gc.collect()

# --- FINAL REPORT ---
print("✅ Label rows loaded:", len(df_labels))
print("✅ Targets:", df_labels["target_id"].nunique())
print("✅ Chains/copies:", df_labels.groupby(["target_id", "chain", "copy"]).ngroups)

print("\n📊 Memory usage (MB):", round(df_labels.memory_usage(deep=True).sum() / 1e6, 2))

df_labels.head(10)

📥 Loading train labels...
📥 Appending validation labels...
✅ Label rows loaded: 7804733
✅ Targets: 5744
✅ Chains/copies: 17952

📊 Memory usage (MB): 1330.14


,resname,chain,copy,target_id,residue_index,x,y,z
0,C,A,1,157D,1,4.843000,-5.640,13.265
1,G,A,1,157D,2,3.385000,-7.613,8.267
2,C,A,1,157D,3,2.158000,-6.751,2.949
3,G,A,1,157D,4,2.669000,-4.843,-1.773
4,A,A,1,157D,5,3.509000,0.239,-4.045
5,A,A,1,157D,6,6.073000,4.823,-5.035
6,U,A,1,157D,7,10.129000,7.706,-4.251
7,U,A,1,157D,8,15.514000,8.745,-2.055
8,A,A,1,157D,9,20.429001,7.281,-1.699
9,G,A,1,157D,10,23.509001,2.728,-1.918


In [3]:
# 3 — ENFORCE SEQUENTIAL GEOMETRY + PHYSICAL STEP FILTER

group_cols = ["target_id", "chain", "copy"]

df_geo = df_labels.copy()

df_geo["resid_diff"] = df_geo.groupby(group_cols)["residue_index"].diff()
df_geo["is_seq"] = df_geo["resid_diff"].eq(1)

df_geo = df_geo[df_geo["resid_diff"].isna() | df_geo["is_seq"]].copy()

df_geo["dx"] = df_geo.groupby(group_cols)["x"].diff()
df_geo["dy"] = df_geo.groupby(group_cols)["y"].diff()
df_geo["dz"] = df_geo.groupby(group_cols)["z"].diff()
df_geo["step"] = np.sqrt(df_geo["dx"]**2 + df_geo["dy"]**2 + df_geo["dz"]**2)

df_geo = df_geo[
    df_geo["step"].isna() | ((df_geo["step"] >= 4.5) & (df_geo["step"] <= 8.0))
].copy()

steps = df_geo["step"].dropna().values
print("✅ Sequential geometry rows:", len(df_geo))
print("✅ Step mean:", float(steps.mean()))
print("✅ Step median:", float(np.median(steps)))
print("✅ Step p90:", float(np.percentile(steps, 90)))
print("✅ Step min:", float(steps.min()))
print("✅ Step max:", float(steps.max()))

display(df_geo.head(10))


✅ Sequential geometry rows: 6996848
✅ Step mean: 5.791539669036865
✅ Step median: 5.574956893920898
✅ Step p90: 7.013174057006836
✅ Step min: 4.500007629394531
✅ Step max: 7.999997615814209


,resname,chain,copy,target_id,residue_index,x,y,z,resid_diff,is_seq,dx,dy,dz,step
0,C,A,1,157D,1,4.843000,-5.640,13.265,NaN,False,NaN,NaN,NaN,NaN
1,G,A,1,157D,2,3.385000,-7.613,8.267,1.0,True,-1.458000,-1.973,-4.998,5.567629
2,C,A,1,157D,3,2.158000,-6.751,2.949,1.0,True,-1.227000,0.862,-5.318,5.525369
3,G,A,1,157D,4,2.669000,-4.843,-1.773,1.0,True,0.511000,1.908,-4.722,5.118483
4,A,A,1,157D,5,3.509000,0.239,-4.045,1.0,True,0.840000,5.082,-2.272,5.629770
5,A,A,1,157D,6,6.073000,4.823,-5.035,1.0,True,2.564000,4.584,-0.990,5.344834
6,U,A,1,157D,7,10.129000,7.706,-4.251,1.0,True,4.056000,2.883,0.784,5.037606
7,U,A,1,157D,8,15.514000,8.745,-2.055,1.0,True,5.385000,1.039,2.196,5.907636
8,A,A,1,157D,9,20.429001,7.281,-1.699,1.0,True,4.915001,-1.464,0.356,5.140746
9,G,A,1,157D,10,23.509001,2.728,-1.918,1.0,True,3.080000,-4.553,-0.219,5.501288


In [4]:
# 4 — BUILD GEOMETRY FEATURES (NORMALIZED DIRECTION + CURVATURE + TURN ANGLE)

df_feat = df_geo.copy()

df_feat["step_safe"] = df_feat["step"].replace(0, np.nan)
df_feat["dx_norm"] = df_feat["dx"] / df_feat["step_safe"]
df_feat["dy_norm"] = df_feat["dy"] / df_feat["step_safe"]
df_feat["dz_norm"] = df_feat["dz"] / df_feat["step_safe"]

df_feat["dx_prev"] = df_feat.groupby(group_cols)["dx"].shift(1)
df_feat["dy_prev"] = df_feat.groupby(group_cols)["dy"].shift(1)
df_feat["dz_prev"] = df_feat.groupby(group_cols)["dz"].shift(1)
df_feat["step_prev"] = df_feat.groupby(group_cols)["step"].shift(1)

df_feat["dx_prev_norm"] = df_feat["dx_prev"] / df_feat["step_prev"]
df_feat["dy_prev_norm"] = df_feat["dy_prev"] / df_feat["step_prev"]
df_feat["dz_prev_norm"] = df_feat["dz_prev"] / df_feat["step_prev"]

df_feat["curvature"] = (
    df_feat["dx_norm"] * df_feat["dx_prev_norm"] +
    df_feat["dy_norm"] * df_feat["dy_prev_norm"] +
    df_feat["dz_norm"] * df_feat["dz_prev_norm"]
)
df_feat["curvature"] = np.clip(df_feat["curvature"], -1.0, 1.0)
df_feat["turn_angle"] = np.arccos(df_feat["curvature"])

feature_cols = ["dx_norm", "dy_norm", "dz_norm", "curvature", "turn_angle", "step"]

df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)
df_feat.to_csv(FEATURE_OUT, index=False)

print("✅ Feature table rows:", len(df_feat))
print("✅ Feature columns:", feature_cols)
print("✅ Saved:", FEATURE_OUT)

display(df_feat[feature_cols].describe())


✅ Feature table rows: 6441646
✅ Feature columns: ['dx_norm', 'dy_norm', 'dz_norm', 'curvature', 'turn_angle', 'step']
✅ Saved: /kaggle/working/FEATURE_TABLE__GEOMETRY_LABELS_V1.csv


,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
count,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06,6.441646e+06
mean,-1.177979e-03,-4.055961e-04,5.236998e-05,6.556832e-01,7.731258e-01,5.791190e+00
std,5.763636e-01,5.767268e-01,5.749628e-01,4.014457e-01,4.991009e-01,7.265077e-01
min,-9.999998e-01,-1.000000e+00,-9.999996e-01,-9.999973e-01,0.000000e+00,4.500008e+00
25%,-5.014891e-01,-5.015000e-01,-4.989068e-01,6.167521e-01,4.604168e-01,5.306649e+00
50%,-3.073370e-03,-1.161220e-03,-1.881477e-04,8.187188e-01,6.116201e-01,5.574335e+00
75%,4.980182e-01,4.998140e-01,4.992287e-01,8.958674e-01,9.061865e-01,6.020129e+00
max,9.999999e-01,1.000000e+00,9.999996e-01,1.000000e+00,3.139277e+00,7.999998e+00


In [5]:
# 5 — LOAD GEOMETRY FEATURE TABLE (DATASET — FINAL, MEMORY SAFE)

import pandas as pd
import gc
import os

# --- DATASET PATH (CONFIRMED STRUCTURE) ---
FEATURE_PATH = "/kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet"

if not os.path.exists(FEATURE_PATH):
    raise FileNotFoundError(f"❌ Feature table not found at:\n{FEATURE_PATH}")

print("📥 Loading geometry feature table...")
print("Path:", FEATURE_PATH)

# --- LOAD PARQUET (COLUMN-OPTIMIZED) ---
USE_COLS = [
    "target_id",
    "chain",
    "copy",
    "residue_index",
    "x", "y", "z",
    "dx", "dy", "dz",
    "dx_norm", "dy_norm", "dz_norm",
    "curvature",
    "turn_angle",
    "step"
]

df_feat = pd.read_parquet(FEATURE_PATH, columns=USE_COLS)

# --- TYPE OPTIMIZATION (CRITICAL FOR MEMORY) ---
float_cols = [
    "x","y","z",
    "dx","dy","dz",
    "dx_norm","dy_norm","dz_norm",
    "curvature","turn_angle","step"
]

for col in float_cols:
    df_feat[col] = df_feat[col].astype("float32")

df_feat["residue_index"] = df_feat["residue_index"].astype("int32")

# --- SORT FOR SEQUENTIAL OPERATIONS ---
df_feat = df_feat.sort_values(
    ["target_id", "chain", "copy", "residue_index"]
).reset_index(drop=True)

# --- BASIC VALIDATION ---
print("\n✅ Feature table loaded")
print("Rows:", len(df_feat))
print("Targets:", df_feat["target_id"].nunique())
print("Chains:", df_feat.groupby(["target_id","chain","copy"]).ngroups)

print("\n📊 Sample:")
display(df_feat.head())

gc.collect()

📥 Loading geometry feature table...
Path: /kaggle/input/datasets/pharaohtut/helix-rna-geometry-v1/FEATURE_TABLE__GEOMETRY_FULL.parquet

✅ Feature table loaded
Rows: 7768829
Targets: 5744
Chains: 17746

📊 Sample:


,target_id,chain,copy,residue_index,x,y,z,dx,dy,dz,dx_norm,dy_norm,dz_norm,curvature,turn_angle,step
0,157D,A,1,2,3.385,-7.613,8.267,-1.458,-1.973,-4.998,-0.261871,-0.354370,-0.897689,0.521913,0.521913,5.567629
1,157D,A,1,3,2.158,-6.751,2.949,-1.227,0.862,-5.318,-0.222067,0.156008,-0.962470,0.392644,0.392644,5.525369
2,157D,A,1,4,2.669,-4.843,-1.773,0.511,1.908,-4.722,0.099834,0.372767,-0.922539,0.761646,0.761646,5.118483
3,157D,A,1,5,3.509,0.239,-4.045,0.840,5.082,-2.272,0.149207,0.902701,-0.403569,0.401361,0.401361,5.629770
4,157D,A,1,6,6.073,4.823,-5.035,2.564,4.584,-0.990,0.479716,0.857651,-0.185226,0.558137,0.558137,5.344834


0

In [6]:
# ============================================================
# DATA LOAD (REQUIRED — FIXES df_test ERROR)
# ============================================================

COMP_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = next((p for p in COMP_ROOT_CANDIDATES if os.path.exists(p)), COMP_ROOT_CANDIDATES[0])

TEST_CSV = os.path.join(COMP_ROOT, "test_sequences.csv")
SAMPLE_SUB_CSV = os.path.join(COMP_ROOT, "sample_submission.csv")

df_test = pd.read_csv(TEST_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)

SUBMISSION_OUT = "/kaggle/working/submission.csv"

print(f"✅ Test targets loaded: {len(df_test)}")

✅ Test targets loaded: 28


In [7]:
# 11 — HAIL MARY V4 (FULL CHAOS + SAFE MERGE + FINAL SUBMIT)

import os
import math
import hashlib
import numpy as np
import pandas as pd

# ============================================================
# DATA LOAD
# ============================================================

COMP_ROOT_CANDIDATES = [
    "/kaggle/input/competitions/stanford-rna-3d-folding-2",
    "/kaggle/input/stanford-rna-3d-folding-2",
]

COMP_ROOT = next((p for p in COMP_ROOT_CANDIDATES if os.path.exists(p)), COMP_ROOT_CANDIDATES[0])

df_test = pd.read_csv(os.path.join(COMP_ROOT, "test_sequences.csv"))
sample_sub = pd.read_csv(os.path.join(COMP_ROOT, "sample_submission.csv"))

SUBMISSION_OUT = "/kaggle/working/submission.csv"

# ============================================================
# CONFIG
# ============================================================

STEP_LEN = 5.82
TARGET_ROG = 45.0

# ============================================================
# UTILS
# ============================================================

def stable_seed(text):
    return int(hashlib.sha256(text.encode()).hexdigest()[:16], 16) % (2**32)

def unit(v):
    n = np.linalg.norm(v)
    if n < 1e-8:
        return np.array([1.0,0.0,0.0])
    return v / n

def build_frame(direction):
    t = unit(direction)
    ref = np.array([0,0,1])
    if abs(np.dot(t,ref)) > 0.9:
        ref = np.array([0,1,0])
    n = unit(np.cross(t,ref))
    b = unit(np.cross(t,n))
    return t,n,b

def radius_of_gyration(coords):
    c = coords.mean(axis=0, keepdims=True)
    return np.sqrt(np.mean(np.sum((coords-c)**2, axis=1)))

def base_bias(base):
    return {
        "turn": {"A":0.1,"C":0.22,"G":0.16,"U":0.28}.get(base,0.18),
        "torsion": {"A":0.06,"C":-0.02,"G":0.08,"U":-0.05}.get(base,0.0),
        "compact": {"A":0.02,"C":0.05,"G":0.03,"U":0.06}.get(base,0.03)
    }

# ============================================================
# CHAOS REGIMES
# ============================================================

REGIMES = [
    {"name":"compact","memory":0.86,"turn":0.65,"torsion":0.25,"mid":0.45,"long":0.20,"repel":0.12,"compact":0.60,"spiral":0.18,"zigzag":0.12,"burst":0.12,"phase":0.30,"freq":0.22},
    {"name":"balanced","memory":0.80,"turn":0.45,"torsion":0.20,"mid":0.30,"long":0.10,"repel":0.10,"compact":0.32,"spiral":0.12,"zigzag":0.10,"burst":0.08,"phase":0.35,"freq":0.28},
    {"name":"expanded","memory":0.74,"turn":0.35,"torsion":0.15,"mid":0.18,"long":0.06,"repel":0.14,"compact":0.08,"spiral":0.25,"zigzag":0.08,"burst":0.16,"phase":0.45,"freq":0.34},
    {"name":"loop","memory":0.82,"turn":0.75,"torsion":0.30,"mid":0.50,"long":0.22,"repel":0.16,"compact":0.42,"spiral":0.15,"zigzag":0.18,"burst":0.14,"phase":0.28,"freq":0.24},
    {"name":"interaction","memory":0.88,"turn":0.55,"torsion":0.28,"mid":0.60,"long":0.30,"repel":0.18,"compact":0.65,"spiral":0.14,"zigzag":0.14,"burst":0.18,"phase":0.20,"freq":0.18},
]

# ============================================================
# GENERATOR
# ============================================================

def generate_chain(target_id, seq, regime, idx):
    rng = np.random.default_rng(stable_seed(f"{target_id}_{idx}_{regime['name']}"))
    L = len(seq)
    coords = np.zeros((L,3))

    if L < 2:
        return coords

    coords[1] = np.array([STEP_LEN,0,0])

    phase = rng.uniform(0, 2*np.pi)
    phase2 = rng.uniform(0, 2*np.pi)
    handed = -1 if idx % 2 == 0 else 1

    for i in range(2,L):
        prev = coords[i-1]
        d1 = unit(coords[i-1]-coords[i-2])
        d0 = unit(coords[i-2]-coords[i-3]) if i>2 else d1

        t,n,b = build_frame(d1)
        bb = base_bias(seq[i-1])

        memory = unit((1+regime["memory"])*d1 - regime["memory"]*d0)

        phase_term = phase + regime["freq"]*i
        phase_term2 = phase2 + regime["freq"]*1.3*i

        zig = regime["zigzag"] * ((-1)**i)

        shape = (
            memory
            + handed*(regime["turn"]+bb["turn"])*n
            + (regime["torsion"]+bb["torsion"])*b
            + regime["phase"]*math.sin(phase_term)*n
            + regime["phase"]*math.cos(phase_term2)*b
            + zig*n
            + regime["spiral"]*(math.sin(phase_term)*n + math.cos(phase_term)*b)
        )

        # interaction
        center = coords[max(0,i-24):i].mean(axis=0)
        interaction = (regime["compact"]+bb["compact"]) * unit(center-prev)

        # mid
        for j in range(max(0,i-20), i-6, 2):
            interaction += regime["mid"] * unit(coords[j]-prev)

        # long
        if i > 10:
            interaction += regime["long"] * unit(coords[i//2]-prev)

        # burst return
        if i > 10 and i % (8+idx) == 0:
            back = max(0, i - rng.integers(6,20))
            interaction += regime["burst"] * unit(coords[back]-prev)

        # repulsion
        repulse = np.zeros(3)
        for j in range(max(0,i-20), i-2):
            d = prev - coords[j]
            dist = np.linalg.norm(d)
            if 1e-8 < dist < 8:
                repulse += (8-dist)/8 * unit(d)
        interaction += regime["repel"] * repulse

        # rog shaping
        rog = radius_of_gyration(coords[:i])
        pull = (TARGET_ROG - rog)/TARGET_ROG
        interaction += 0.18 * pull * unit(center-prev)

        direction = unit(shape + interaction)

        if i > 6:
            recent = [unit(coords[k]-coords[k-1]) for k in range(max(2,i-5), i)]
            if np.mean([np.dot(direction,d) for d in recent]) > 0.965:
                direction = unit(direction + 0.6*n + 0.35*b)

        coords[i] = prev + STEP_LEN * direction

    return coords

def generate_ensemble(tid, seq):
    return np.stack([generate_chain(tid, seq, r, i) for i,r in enumerate(REGIMES)])

# ============================================================
# BUILD SUBMISSION
# ============================================================

rows = []
for row in df_test.itertuples(index=False):
    coords = generate_ensemble(row.target_id, row.sequence)
    for i in range(len(row.sequence)):
        r = {"ID":f"{row.target_id}_{i+1}","resname":row.sequence[i],"resid":i+1}
        for s in range(5):
            r[f"x_{s+1}"], r[f"y_{s+1}"], r[f"z_{s+1}"] = coords[s,i]
        rows.append(r)

submission = pd.DataFrame(rows)

# ============================================================
# SAFE MERGE
# ============================================================

merged = sample_sub.merge(
    submission,
    on=["ID","resname","resid"],
    how="left",
    suffixes=("_orig","")
)

merged = merged.drop(columns=[c for c in merged.columns if c.endswith("_orig")])

expected_cols = ["ID","resname","resid"] + [f"{a}_{i}" for i in range(1,6) for a in ["x","y","z"]]

submission = merged[expected_cols].copy()

coord_cols = [c for c in submission.columns if c.startswith(("x_","y_","z_"))]
submission[coord_cols] = submission[coord_cols].replace([np.inf,-np.inf],0).fillna(0).astype(np.float32)

submission.to_csv(SUBMISSION_OUT, index=False)

print("🔥 HAIL MARY V4 COMPLETE")
print("📁 Saved:", SUBMISSION_OUT)
print("Shape:", submission.shape)

🔥 HAIL MARY V4 COMPLETE
📁 Saved: /kaggle/working/submission.csv
Shape: (9762, 18)


In [8]:
# 12 — VALIDATE AND SAVE FINAL SUBMISSION (STRICT — SCHEMA LOCKED)

import numpy as np
import pandas as pd

OUT_PATH = "/kaggle/working/submission.csv"

print("🔍 Running strict submission validation...")

sample = pd.read_csv(SAMPLE_SUB_PATH)

# --- HARD SCHEMA LOCK ---
expected_cols = list(sample.columns)

missing_cols = [c for c in expected_cols if c not in submission.columns]
if missing_cols:
    raise ValueError(f"❌ Submission is missing required columns: {missing_cols}")

extra_cols = [c for c in submission.columns if c not in expected_cols]
if extra_cols:
    print(f"⚠️ Dropping extra columns not allowed by sample_submission: {extra_cols}")

# Keep only exact sample columns and exact order
submission = submission.loc[:, expected_cols].copy()

# --- BASIC STRUCTURE CHECKS ---
if list(submission.columns) != expected_cols:
    raise ValueError(
        "❌ Column mismatch after schema lock\n\n"
        f"Expected:\n{expected_cols}\n\n"
        f"Got:\n{list(submission.columns)}"
    )

if len(submission) != len(sample):
    raise ValueError(
        f"❌ Row count mismatch: expected {len(sample)}, got {len(submission)}"
    )

# --- ID VALIDATION ---
sub_ids = submission["ID"].astype(str).str.strip().reset_index(drop=True)
sample_ids = sample["ID"].astype(str).str.strip().reset_index(drop=True)

if not sub_ids.equals(sample_ids):
    mismatches = (sub_ids != sample_ids)
    mismatch_indices = np.where(mismatches)[0][:10]

    debug_rows = pd.DataFrame({
        "submission_ID": sub_ids.iloc[mismatch_indices].tolist(),
        "sample_ID": sample_ids.iloc[mismatch_indices].tolist(),
    })

    raise ValueError(
        "❌ Submission IDs do NOT match sample_submission.csv\n\n"
        f"First mismatches:\n{debug_rows}"
    )

print("✅ ID alignment PASSED")

# --- COORDINATE VALIDATION ---
coord_cols = [c for c in submission.columns if c.startswith(("x_", "y_", "z_"))]

if len(coord_cols) == 0:
    raise ValueError("❌ No coordinate columns found (x_*, y_*, z_*)")

# Force numeric before validation
for c in coord_cols:
    submission[c] = pd.to_numeric(submission[c], errors="raise")

if submission[coord_cols].isnull().any().any():
    bad = submission[coord_cols].isnull().sum()
    raise ValueError(f"❌ Null values detected:\n{bad[bad > 0]}")

vals = submission[coord_cols].to_numpy(dtype=np.float64)
if not np.isfinite(vals).all():
    raise ValueError("❌ Non-finite values found (NaN or Inf)")

print("✅ Coordinate validity PASSED")

# --- TYPE ENFORCEMENT ---
submission["resname"] = submission["resname"].astype(str)
submission["resid"] = pd.to_numeric(submission["resid"], errors="raise").astype(np.int32)
for c in coord_cols:
    submission[c] = submission[c].astype(np.float32)

print("✅ Dtype enforcement PASSED")

# --- FINAL SAVE ---
submission.to_csv(OUT_PATH, index=False)

print("\n🚀 SUBMISSION READY")
print("📁 Saved to:", OUT_PATH)
print("📐 Shape:", submission.shape)
print("📊 Coordinate dtype sample:")
print(submission[coord_cols].dtypes.head())

🔍 Running strict submission validation...
✅ ID alignment PASSED
✅ Coordinate validity PASSED
✅ Dtype enforcement PASSED

🚀 SUBMISSION READY
📁 Saved to: /kaggle/working/submission.csv
📐 Shape: (9762, 18)
📊 Coordinate dtype sample:
x_1    float32
y_1    float32
z_1    float32
x_2    float32
y_2    float32
dtype: object


In [9]:
# DIAG_BRIDGE_UNIVERSAL — DATASET-AGNOSTIC BRIDGE OUTPUT ENGINE

import os
import json
import numpy as np
import pandas as pd

OUTPUT_ROOT = "./bridge_outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

GEOMETRY_OUT = os.path.join(OUTPUT_ROOT, "BRIDGE_GEOMETRY_UNITS.csv")
METHOD_OUT   = os.path.join(OUTPUT_ROOT, "BRIDGE_METHOD_OUTPUTS.csv")
REALITY_OUT  = os.path.join(OUTPUT_ROOT, "BRIDGE_STRUCTURAL_REALITY.csv")
MANIFEST_OUT = os.path.join(OUTPUT_ROOT, "GENERIC_RUN_MANIFEST.json")

# ============================================================
# 1 — INPUT DETECTION (FLEXIBLE)
# ============================================================

df_input = None

CANDIDATES = ["submission", "df_submission", "df_coords", "df"]

for name in CANDIDATES:
    if name in globals() and isinstance(globals()[name], pd.DataFrame):
        df_input = globals()[name].copy()
        print(f"✅ Using input dataframe: {name}")
        break

if df_input is None:
    raise RuntimeError("❌ No valid dataframe found in globals()")

# ============================================================
# 2 — COLUMN NORMALIZATION
# ============================================================

if "ID" in df_input.columns:
    df_input["target_id"] = df_input["ID"].astype(str).str.rsplit("_", n=1).str[0]
    df_input["residue_index"] = pd.to_numeric(
        df_input["ID"].astype(str).str.rsplit("_", n=1).str[-1],
        errors="coerce"
    )
else:
    raise RuntimeError("❌ ID column required")

df_input = df_input.dropna(subset=["residue_index"]).copy()
df_input["residue_index"] = df_input["residue_index"].astype(int)

# detect coordinate sets
coord_sets = []
for i in range(1, 6):
    if f"x_{i}" in df_input.columns:
        coord_sets.append((f"x_{i}", f"y_{i}", f"z_{i}"))

if len(coord_sets) == 0:
    if {"x","y","z"}.issubset(df_input.columns):
        coord_sets = [("x","y","z")]
    else:
        raise RuntimeError("❌ No coordinate columns found")

print(f"✅ Detected {len(coord_sets)} coordinate tracks")

# ============================================================
# 3 — GEOMETRY UNITS
# ============================================================

geometry_rows = []

for (xcol, ycol, zcol) in coord_sets:
    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 2:
            continue

        deltas = xyz[1:] - xyz[:-1]
        steps = np.linalg.norm(deltas, axis=1)

        for i in range(len(steps)):
            geometry_rows.append({
                "target_id": tid,
                "residue_index": int(grp["residue_index"].iloc[i]),
                "step": float(steps[i]),
                "dx": float(deltas[i][0]),
                "dy": float(deltas[i][1]),
                "dz": float(deltas[i][2]),
                "track": xcol
            })

df_geometry = pd.DataFrame(geometry_rows)
df_geometry.to_csv(GEOMETRY_OUT, index=False)

print("✅ Geometry units written:", GEOMETRY_OUT)

# ============================================================
# 4 — METHOD OUTPUTS (PER TRACK SUMMARY)
# ============================================================

method_rows = []

for (xcol, ycol, zcol) in coord_sets:
    steps_all = []
    rog_all = []

    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 2:
            continue

        steps = np.linalg.norm(xyz[1:] - xyz[:-1], axis=1)
        steps_all.extend(steps)

        center = xyz.mean(axis=0)
        rog = np.sqrt(np.mean(np.sum((xyz - center)**2, axis=1)))
        rog_all.append(rog)

    method_rows.append({
        "track": xcol,
        "mean_step": float(np.mean(steps_all)),
        "std_step": float(np.std(steps_all)),
        "mean_rog": float(np.mean(rog_all)),
    })

df_method = pd.DataFrame(method_rows)
df_method.to_csv(METHOD_OUT, index=False)

print("✅ Method outputs written:", METHOD_OUT)

# ============================================================
# 5 — STRUCTURAL REALITY
# ============================================================

reality_rows = []

for (xcol, ycol, zcol) in coord_sets:
    for tid, grp in df_input.groupby("target_id"):
        grp = grp.sort_values("residue_index")
        xyz = grp[[xcol, ycol, zcol]].values.astype(np.float64)

        if len(xyz) < 3:
            continue

        steps = np.linalg.norm(xyz[1:] - xyz[:-1], axis=1)
        center = xyz.mean(axis=0)
        rog = np.sqrt(np.mean(np.sum((xyz - center)**2, axis=1)))

        status = "VALID"

        if np.std(steps) > 1.5:
            status = "STEP_UNSTABLE"
        elif rog < 5:
            status = "COLLAPSED"
        elif rog > 300:
            status = "EXPLODED"

        reality_rows.append({
            "target_id": tid,
            "track": xcol,
            "mean_step": float(np.mean(steps)),
            "std_step": float(np.std(steps)),
            "radius_of_gyration": float(rog),
            "status": status
        })

df_reality = pd.DataFrame(reality_rows)
df_reality.to_csv(REALITY_OUT, index=False)

print("✅ Structural reality written:", REALITY_OUT)

# ============================================================
# 6 — MANIFEST
# ============================================================

manifest = {
    "rows_input": int(len(df_input)),
    "targets": int(df_input["target_id"].nunique()),
    "tracks": len(coord_sets),
    "geometry_rows": int(len(df_geometry)),
    "status": "COMPLETE"
}

with open(MANIFEST_OUT, "w") as f:
    json.dump(manifest, f, indent=2)

print("✅ Manifest written:", MANIFEST_OUT)

print("\n" + "="*60)
print("🚀 UNIVERSAL BRIDGE COMPLETE")
print("="*60)

✅ Using input dataframe: submission
✅ Detected 5 coordinate tracks
✅ Geometry units written: ./bridge_outputs/BRIDGE_GEOMETRY_UNITS.csv
✅ Method outputs written: ./bridge_outputs/BRIDGE_METHOD_OUTPUTS.csv
✅ Structural reality written: ./bridge_outputs/BRIDGE_STRUCTURAL_REALITY.csv
✅ Manifest written: ./bridge_outputs/GENERIC_RUN_MANIFEST.json

🚀 UNIVERSAL BRIDGE COMPLETE
